In [1]:
%load_ext autoreload
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

# General

In [2]:
from ioutils import FileIO
from fileutils import DirInfo, FileInfo
from master import MasterParams, MusicDBPermDir
from pandas import Series, DataFrame, concat
from sys import prefix
from listUtils import getFlatList
from musicdb import PanDBIO
mp    = MasterParams(verbose=True)
io    = FileIO()
mdbpd = MusicDBPermDir()

MasterBasic()
  ==> ModVals:    100
  ==> Project:    pandb
  ==> MusicDB:    musicdb
MasterPaths()
  ==> Raw:        /Volumes/Piggy/Discog
  ==> Mod:        /Volumes/Seagate/Discog
  ==> Sum:        /Users/tgadfort/Music/Discog
MasterMetas()
  ==> Media:      ['Album', 'SingleEP', 'Appearance', 'Technical', 'Mix', 'Bootleg', 'AltMedia', 'Other']
  ==> Metas:      ['Basic', 'Media', 'Genre', 'Bio', 'Link', 'Metric', 'Counts']
  ==> Searches:   ['Name', 'AlbumMedia', 'SingleEPMedia', 'AppearanceMedia', 'TechnicalMedia', 'MixMedia', 'BootlegMedia', 'AltMediaMedia', 'OtherMedia']
MasterDBs()
  ==> DBs:        ['Discogs', 'Spotify', 'LastFM', 'Genius', 'RateYourMusic', 'MetalArchives', 'Deezer', 'AllMusic', 'MusicBrainz', 'AlbumOfTheYear', 'SetListFM', 'Beatport', 'Traxsource']


# DB-Specific

In [3]:
from lib import traxsource
mio   = traxsource.MusicDBIO(verbose=False)
webio = traxsource.RawWebData()
db    = mio.db
permDBDir = mdbpd.getDBPermPath(db)
print("Saving Perminant {0} DB Data To {1}".format(db, permDBDir.str))

Saving Perminant Traxsource DB Data To /Users/tgadfort/opt/anaconda3/envs/py310/pandb/mdblib/Traxsource


# Master Files

In [4]:
if False:
    starter = io.get("starter.p")
    starter = starter[starter["Name"].str.len() > 0]
    starter.index = starter["URL"].map(mio.getdbid)    
    mio.data.saveSearchArtistData(data=starter)

In [5]:
from base import MusicDBDir, MusicDBData
permDir = MusicDBDir(permDBDir)
localArtists       = MusicDBData(path=permDir, fname="{0}SearchedForLocalArtists".format(db.lower()))
localXtraArtists   = MusicDBData(path=permDir, fname="{0}SearchedForLocalXtraArtists".format(db.lower()))
masterArtists      = MusicDBData(path=permDir, fname="{0}SearchedForMasterArtists".format(db.lower()))
searchArtists      = mio.data.getSearchArtistData()
knownArtists       = mio.data.getSummaryNameData()
errors             = MusicDBData(path=permDir, fname="{0}SearchedForErrors".format(db.lower()))

In [6]:
##########################################################################################
# Show Summary
##########################################################################################
print("{0} Search Results".format(db))
print("   Local Artists:     {0}".format(len(localArtists.get())))
print("   Local XtraArtists: {0}".format(len(localXtraArtists.get())))
print("   Master Artists:    {0}".format(len(masterArtists.get())))
print("   Errors:            {0}".format(len(errors.get())))
print("   Search Artists:    {0}".format(searchArtists.shape[0]))
print("   Known IDs:         {0}".format(len(knownArtists)))

Traxsource Search Results
   Local Artists:     71172
   Local XtraArtists: 6190
   Master Artists:    0
   Errors:            4
   Search Artists:    137851
   Known IDs:         53045


# Download Artist Data

In [7]:
mio   = traxsource.MusicDBIO(verbose=False,local=True,mkDirs=False)
webio = traxsource.RawWebData(debug=False)

In [25]:
useSearchData = True

if useSearchData is True:
    artistNames      = searchArtists #.sort_values(by="Num", ascending=False)
    localArtistsDict = localArtists.get()
    artistNamesToGet = artistNames[~artistNames.index.isin(localArtistsDict.keys())].sample(frac=1)

    print("{0} Search Results".format(db))
    print("   Available Names:      {0}".format(len(artistNames)))
    print("   Known Artist Names:   {0}".format(len(localArtistsDict)))
    print("   Artist Names To Get:  {0}".format(len(artistNamesToGet)))
    
#   Artist Names To Get:  71955
#   Artist Names To Get:  61780
#   Artist Names To Get:  40905
#   Artist Names To Get:  26030

Traxsource Search Results
   Available Names:      137851
   Known Artist Names:   117097
   Artist Names To Get:  20755


In [ ]:
from timeutils import Timestat, TermTime

ts = Timestat("Getting {0} ArtistIDs".format(db))
tt = TermTime("tomorrow", "10:50am")
#tt = TermTime("today", "9:00pm")
maxN = 50000000

n  = 0
localArtistsDict  = localArtists.get()
searchedForErrors = errors.get()

for i,(artistID,row) in enumerate(artistNamesToGet.iterrows()):
    artistName = row["Name"]
    artistURL  = row["Ref"]
    if localArtistsDict.get(artistID) is not None:
        continue
    if searchedForErrors.get(artistID) is not None:
        continue
        
    response = webio.getArtistData(artistName=artistName, artistURL=artistURL)
    if response is None or len(response) == 0:
        print("Error ==> {0}".format((artistID,artistName)))
        searchedForErrors[artistID] = artistName
        errors.save(data=searchedForErrors)
        webio.sleep(5)
        continue
        
    mio.data.saveRawData(data=response, modval=mio.getModVal(artistID), dbID=artistID)
    localArtistsDict[artistID] = artistName
    webio.sleep(5.0)
    n += 1
        
    if n % 25 == 0:
        print("="*150)
        ts.update(n=n)
        print("Saving {0} {1} Artists Data".format(len(localArtistsDict), db))
        localArtists.save(data=localArtistsDict)
        print("="*150)
        webio.wait(5.0)
        if tt.isFinished():
            break
    
ts.stop()
print("Saving {0} {1} Artists Data".format(len(localArtistsDict), db))
localArtists.save(data=localArtistsDict)
if len(searchedForErrors) > 0:
    print("Saving {0} {1} Errors".format(len(searchedForErrors), db))
    errors.save(data=searchedForErrors)

Process [Getting Traxsource ArtistIDs] Start    ==> Time Is 2022-04-23 22:23:22
========================= termTime(day=tomorrow,time=10:50am) =========================
   ====> Terminate Time Set To 2022-04-24 10:50:00 <====
   ====> Will Terminate Process 12 Hours and 26 Minutes From Now
Getting Data For True Blood (459557) P=1                           True
Getting Data For Petre Inspirescu (56600) P=1                      True
Getting Data For Soulskid (26580) P=1                              True
Getting Data For Room With A View (322743) P=1                     True
Getting Data For D'Anthony (409805) P=1                            True
Getting Data For Krucy Darkstar (158442) P=1                       True
Getting Data For Shantelle (375469) P=1                            True
Getting Data For saib. (420466) P=1                                True
Getting Data For Amo & Navas (97201) P=1                           True
Getting Data For Wotsisname (109987) P=1                      

Waiting:   0%|          | 0/50 [00:00<?, ?it/s]

Getting Data For Stefano Albanese (63801) P=1                      True
Getting Data For Icarus in Love (426283) P=1                       True
Getting Data For Pau Guilera (590595) P=1                          True
Getting Data For Elvin T (524698) P=1                              True
Getting Data For You & Oui (232560) P=1                            True
Getting Data For Miss Min.D (518526) P=1                           True
Getting Data For Disco Superstars (583134) P=1                     True
Getting Data For Mana Traya (330929) P=1                           True
Getting Data For Burty (274588) P=1                                True
Getting Data For Lorena Llanes (95506) P=1                         True
Getting Data For Blaark (360505) P=1                               True
Getting Data For Damian Fernandez (19156) P=1                      True
Getting Data For Dio Zambrano (96925) P=1                          True
Getting Data For Diet Candy (394246) P=1                        

Waiting:   0%|          | 0/50 [00:00<?, ?it/s]

Getting Data For AL-X (38827) P=1                                  True
Getting Data For Laolu Remix (648931) P=1                          True
Getting Data For Boris Castro (29494) P=1                          True
Getting Data For Simon Peter (227909) P=1                          True
Getting Data For Yannick Labbé (189983) P=1                       True
Getting Data For Sexyhard Beats (218975) P=1                       True
Getting Data For J Boogie (546) P=1                                True
Getting Data For Bobby Tee (20003) P=1                             True
Getting Data For Plot Pilot (559474) P=1                           True
Getting Data For Operator (45647) P=1                              True
Getting Data For MARKHAM (141965) P=1                              True
Getting Data For Sergent Garcia (50294) P=1                        True
Getting Data For August (21212) P=1                                True
Getting Data For U-GEEN (593103) P=1                            

...

Getting Data For Amy Allen (583360) P=1                            True
Getting Data For Cedric Scheibel (449313) P=1                      True
Getting Data For REC (27410) P=1                                   True
Getting Data For Holotype (395384) P=1                             True
Getting Data For Extra Medium feat. Mr Switch & Cab Canavaral (426336) P=1True
Getting Data For Seamus Haji & Romain Curtis Feat. Awa (313843) P=1True
Getting Data For Lumberjack (106866) P=1                           True
Getting Data For The Delta (141845) P=1                            True
Getting Data For John Wick (526917) P=1                            True
Getting Data For Milos Drvenica (144816) P=1                       True
Getting Data For Lostcause (129891) P=1                            True
Getting Data For Mechanical Soul Brother (113231) P=1              True
Getting Data For Zac Graham (494679) P=1                           True
Getting Data For Derry Kost (234125) P=1                 

Waiting:   0%|          | 0/50 [00:00<?, ?it/s]

Getting Data For Fran Mahema (133504) P=1                          True
Getting Data For Jay Pei (193636) P=1                              True
Getting Data For Josh From Blaze (330181) P=1                      True
Getting Data For Luke Terry (171100) P=1                           True
Getting Data For Wilma (AU) (636880) P=1                           True
Getting Data For Guzy (367533) P=1                                 True
Getting Data For Cury (361196) P=1                                 True
Getting Data For Mack Moses (427617) P=1                           True
Getting Data For Maxine Garman (338263) P=1                        True
Getting Data For Macam (334705) P=1                                True
Getting Data For Mawimbi (285755) P=1                              True
Getting Data For Bredes Fernando (85497) P=1                       True
Getting Data For Favila (280691) P=1                               True
Getting Data For Akua Naru (147080) P=1                         

Waiting:   0%|          | 0/50 [00:00<?, ?it/s]

Getting Data For Sandecki (101040) P=1                             True


In [12]:
del searchedForErrors['296602']
errors.save(data=searchedForErrors)

In [ ]:
from lib.traxsource import moveLocalFiles
moveLocalFiles()

# Download Extra Artist Files

In [ ]:
mio   = traxsource.MusicDBIO(verbose=False,local=True,mkDirs=False)
webio = traxsource.RawWebData(debug=False)

In [ ]:
artistNames = None
for modVal in range(100):
    modValData = mio.data.getModValData(modVal)
    modValPagesData = modValData.apply(lambda rData: (rData.pages.tot, rData.artist.name, rData.url.url)).apply(Series)    
    artistNames = concat([artistNames, modValPagesData[modValPagesData[0] > 1]]) if isinstance(modValPagesData,DataFrame) else  modValPagesData[modValPagesData[0] > 1]
artistNames.columns = ["Pages", "Name", "URL"]
artistNames["URL"] = artistNames["URL"].apply(lambda url: url[26:])
artistNames = artistNames.sort_values("Pages", ascending=False)

localXtraArtistsDict = localXtraArtists.get()
artistNamesToGet = artistNames[~artistNames.index.isin(localXtraArtistsDict.keys())]
print("{0} Search Results".format(db))
print("   Available Names:      {0}".format(len(artistNames)))
print("   Known Artist Names:   {0}".format(len(localXtraArtistsDict)))
print("   Artist Names To Get:  {0}".format(len(artistNamesToGet)))
    
#   Artist Names To Get:  1457
#   Artist Names To Get:  4665
#   Artist Names To Get:  3890
#   Artist Names To Get:  1040

In [ ]:
from timeutils import Timestat, TermTime

ts = Timestat("Getting {0} ArtistIDs".format(db))
tt = TermTime("tomorrow", "6:50am")
#tt = TermTime("today", "9:00pm")
#tt = TermTime("today", "11:06am")
maxN = 50000000

n  = 0
localXtraArtistsDict = localXtraArtists.get()
searchedForErrors = errors.get()

for i,(artistID,row) in enumerate(artistNamesToGet.iterrows()):
    artistName  = row["Name"]
    artistURL   = row["URL"]
    artistPages = row["Pages"]
    if localXtraArtistsDict.get(artistID) is not None:
        continue
    if searchedForErrors.get(artistID) is not None:
        continue
        
    good = True
    for page in range(2,artistPages+1):
        response = webio.getArtistData(artistName=artistName, artistURL=artistURL, page=page)
        if response is None or len(response) == 0:
            print("Error ==> {0}".format((artistID,artistName)))
            searchedForErrors[artistID] = artistName
            errors.save(data=searchedForErrors)
            webio.sleep(5.0)
            good = False
            break

        mio.data.saveRawArtistExtraData(data=response, modval=mio.getModVal(artistID), dbID="{0}-{1}".format(artistID,page))
        webio.sleep(5.0)

    if good is True:
        localXtraArtistsDict[artistID] = artistName        
        n += 1
        
    if n % 5 == 0:
        print("="*150)
        ts.update(n=n)
        print("Saving {0} {1} Artists Data".format(len(localXtraArtistsDict), db))
        localXtraArtists.save(data=localXtraArtistsDict)
        print("="*150)
        webio.wait(5.0)
        if tt.isFinished():
            break
    
ts.stop()
print("Saving {0} {1} Artists Data".format(len(localXtraArtistsDict), db))
localXtraArtists.save(data=localXtraArtistsDict)
if len(searchedForErrors) > 0:
    print("Saving {0} {1} Errors".format(len(searchedForErrors), db))
    errors.save(data=searchedForErrors)

# Create Starter

In [ ]:
from collections import Counter
from ioutils import FileIO, HTMLIO
from timeutils import Timestat
io = FileIO()
hio = HTMLIO()
cntr = Counter()
refs = []
mio   = traxsource.MusicDBIO()
ts = Timestat("Creating Starter")
for modVal in range(100):
    for ifile in mio.dir.getRawModValDataDir(modVal).glob("*.p", debug=False):
        bsdata = hio.get(io.get(ifile))
        refs  += [{"Name": ref.text, "URL": ref.get('href')} for ref in bsdata.findAll("a") if ref.attrs['href'].startswith("/artist/")]
    ts.update(n=modVal+1, N=100)
    if (modVal+1) % 5 == 0:
        print(modVal+1,'\t',len(refs))
ts.stop()    

In [ ]:
df = DataFrame(refs)
df.index = df["URL"].map(mio.getdbid)
N  = df.index.value_counts()
N.name = "Num"
df = df.drop_duplicates(subset=['URL'])
df.index.name = None
df.columns = ["Name", "Ref"]
df = df.join(N)
df = df.sort_values(by='Num', ascending=False)
mio.data.saveSearchArtistData(data=df)

# Move Local

In [ ]:
from lib.traxsource import moveLocalFiles

In [ ]:
moveLocalFiles(verbose=True)

In [ ]:
from utils import PoolIO
pio = PoolIO("Traxsource")
pio.parse()
pio.merge()
pio.meta()
pio.sum()
pio.search()